### Step 1: Data Loading and Initial Preparation
In this step, we import the necessary libraries, load the raw healthcare CSV, and perform initial cleaning. We standardize column names to lowercase with underscores to make them compatible with SQL and calculate the 'length of stay' for each patient.

In [10]:
# CELL 1: Load and initial data prep
import pandas as pd
import numpy as np

# Load the healthcare dataset from CSV
df = pd.read_csv('/content/healthcare_dataset.csv')

# Show basic info and check for missing data
print("--- Dataset Info ---")
print(df.info())

# Clean column names and convert text dates into actual date objects
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')
df['date_of_admission'] = pd.to_datetime(df['date_of_admission'])
df['discharge_date'] = pd.to_datetime(df['discharge_date'])

# Create a new column to see how many days each patient stayed
df['length_of_stay'] = (df['discharge_date'] - df['date_of_admission']).dt.days

df.head()

--- Dataset Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 55500 entries, 0 to 55499
Data columns (total 15 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Name                55500 non-null  object 
 1   Age                 55500 non-null  int64  
 2   Gender              55500 non-null  object 
 3   Blood Type          55500 non-null  object 
 4   Medical Condition   55500 non-null  object 
 5   Date of Admission   55500 non-null  object 
 6   Doctor              55500 non-null  object 
 7   Hospital            55500 non-null  object 
 8   Insurance Provider  55500 non-null  object 
 9   Billing Amount      55500 non-null  float64
 10  Room Number         55500 non-null  int64  
 11  Admission Type      55500 non-null  object 
 12  Discharge Date      55500 non-null  object 
 13  Medication          55500 non-null  object 
 14  Test Results        55500 non-null  object 
dtypes: float64(1), int64(2), object(

,name,age,gender,blood_type,medical_condition,date_of_admission,doctor,hospital,insurance_provider,billing_amount,room_number,admission_type,discharge_date,medication,test_results,length_of_stay
0,Bobby JacksOn,30,Male,B-,Cancer,2024-01-31,Matthew Smith,Sons and Miller,Blue Cross,18856.281306,328,Urgent,2024-02-02,Paracetamol,Normal,2
1,LesLie TErRy,62,Male,A+,Obesity,2019-08-20,Samantha Davies,Kim Inc,Medicare,33643.327287,265,Emergency,2019-08-26,Ibuprofen,Inconclusive,6
2,DaNnY sMitH,76,Female,A-,Obesity,2022-09-22,Tiffany Mitchell,Cook PLC,Aetna,27955.096079,205,Emergency,2022-10-07,Aspirin,Normal,15
3,andrEw waTtS,28,Female,O+,Diabetes,2020-11-18,Kevin Wells,"Hernandez Rogers and Vang,",Medicare,37909.782410,450,Elective,2020-12-18,Ibuprofen,Abnormal,30
4,adrIENNE bEll,43,Female,AB+,Cancer,2022-09-19,Kathleen Hanna,White-White,Aetna,14238.317814,458,Urgent,2022-10-09,Penicillin,Abnormal,20


### Step 2: Data Formatting and Indexing
Here, we refine the data by standardizing the name casing to 'Title Case', rounding financial billing amounts to two decimal places, and generating a unique 'admission_id' to serve as a primary key.

In [11]:
# CELL 2: Data Formatting
# Make names look neat (Title Case), round bills to 2 decimals, and add a unique ID for each patient
df['name'] = df['name'].str.title()
df['billing_amount'] = df['billing_amount'].round(2)
df.insert(0, 'admission_id', range(1001, 1001 + len(df)))

# Check if any rows are exact duplicates
print(f"Number of duplicate rows: {df.duplicated().sum()}")
df.head()

Number of duplicate rows: 0


,admission_id,name,age,gender,blood_type,medical_condition,date_of_admission,doctor,hospital,insurance_provider,billing_amount,room_number,admission_type,discharge_date,medication,test_results,length_of_stay
0,1001,Bobby Jackson,30,Male,B-,Cancer,2024-01-31,Matthew Smith,Sons and Miller,Blue Cross,18856.28,328,Urgent,2024-02-02,Paracetamol,Normal,2
1,1002,Leslie Terry,62,Male,A+,Obesity,2019-08-20,Samantha Davies,Kim Inc,Medicare,33643.33,265,Emergency,2019-08-26,Ibuprofen,Inconclusive,6
2,1003,Danny Smith,76,Female,A-,Obesity,2022-09-22,Tiffany Mitchell,Cook PLC,Aetna,27955.10,205,Emergency,2022-10-07,Aspirin,Normal,15
3,1004,Andrew Watts,28,Female,O+,Diabetes,2020-11-18,Kevin Wells,"Hernandez Rogers and Vang,",Medicare,37909.78,450,Elective,2020-12-18,Ibuprofen,Abnormal,30
4,1005,Adrienne Bell,43,Female,AB+,Cancer,2022-09-19,Kathleen Hanna,White-White,Aetna,14238.32,458,Urgent,2022-10-09,Penicillin,Abnormal,20


### Step 3: SQLite Database Integration
To enable advanced querying, we create a local SQLite database named `healthcare.db` and save the cleaned DataFrame into a table called `hospital_data`.

In [12]:
# CELL 3: Save to Database
import sqlite3

# Create a local SQL database and save our cleaned data into a table called 'hospital_data'
conn = sqlite3.connect('healthcare.db')
df.to_sql('hospital_data', conn, if_exists='replace', index=False)

print("Success: Data loaded into SQL 'hospital_data' table!")

Success: Data loaded into SQL 'hospital_data' table!


### Step 4: SQL Revenue Analysis
We run a SQL query against our database to identify the top 10 combinations of insurance providers and medical conditions based on total billing revenue.

In [13]:
# CELL 4: SQL Analysis
# Run a query to find which insurance companies and conditions generate the most revenue
query = """
SELECT
    insurance_provider,
    medical_condition,
    COUNT(*) as total_patients,
    ROUND(SUM(billing_amount), 2) as total_revenue
FROM hospital_data
GROUP BY insurance_provider, medical_condition
ORDER BY total_revenue DESC
LIMIT 10;
"""

revenue_report = pd.read_sql(query, conn)
revenue_report

,insurance_provider,medical_condition,total_patients,total_revenue
0,Blue Cross,Obesity,1891,49356584.80
1,Medicare,Diabetes,1903,48852772.87
2,Cigna,Asthma,1907,48839168.73
3,Cigna,Obesity,1864,48682086.58
4,Aetna,Hypertension,1876,48581298.74
5,Cigna,Diabetes,1893,48557738.11
6,UnitedHealthcare,Asthma,1870,48282715.15
7,Cigna,Arthritis,1900,48104529.67
8,Blue Cross,Diabetes,1860,48025287.93
9,UnitedHealthcare,Arthritis,1873,48000288.72


### Step 5: Statistical Summary
This cell provides a quick statistical overview of the billing amounts and stay lengths to help identify any potential outliers or data entry errors.

In [14]:
# CELL 5: Quick Stats
# Check the minimum, maximum, and average billing amounts to spot outliers
df.agg({'billing_amount': ['min', 'max', 'mean'], 'length_of_stay': ['min', 'max']})

,billing_amount,length_of_stay
min,-2008.490000,1.0
max,52764.280000,30.0
mean,25539.316071,NaN


### Step 6: Final Cleanup and Export
In the final step, we filter out any invalid records (like negative bills), perform a final integrity check, and export the 'gold standard' dataset as a CSV file for download.

In [15]:
# CELL 6: Final Cleanup and Export
# Remove any records with zero or negative bills, verify data integrity, and download the final file
df = df[df['billing_amount'] > 0].copy()

print("--- Final Data Integrity Check ---")
print(f"Total Records: {len(df)}")
print(f"Any Nulls?: {df.isnull().values.any()}")

# Save to CSV and download to your computer
df.to_csv('gold_standard_healthcare_data.csv', index=False)
from google.colab import files
files.download('gold_standard_healthcare_data.csv')

--- Final Data Integrity Check ---
Total Records: 55392
Any Nulls?: False


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>